# Phase 2 — NLP Classification of Digital Contracts

**Deliverable L2:** Annotated corpus + evaluated classifier + full-corpus propagation.

**Prerequisites (run in order before this notebook):**
```
python scripts/nlp_annotation_sample.py
# → fill data/annotation/boamp_annotation_gold.csv
python scripts/nlp_baseline_classifier.py
python scripts/nlp_advanced_classifier.py
python scripts/nlp_propagate_labels.py
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
FIGURES = Path('../reports/figures/nlp')
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES = Path('../reports/tables/nlp')

CATEGORY_LABELS = {
    'CAT01': 'Software & Apps',
    'CAT02': 'IT Services',
    'CAT03': 'Cybersecurity',
    'CAT04': 'Telecom & Networks',
    'CAT05': 'Cloud & Infra',
    'CAT06': 'IT Hardware',
    'CAT07': 'Digital Workplace',
    'CAT08': 'Data & AI',
    'CAT09': 'GIS & Mapping',
    'CAT10': 'IT Maintenance',
    'CAT_UNKNOWN': 'Unknown',
}
print('Setup done')

## 1. Annotation Statistics

In [ ]:
gold = pd.read_csv('../data/annotation/boamp_annotation_gold.csv')
gold['category_id_final'] = gold['category_id_final'].str.strip()
gold = gold[gold['category_id_final'].notna() & (gold['category_id_final'] != '')]

print(f'Annotated rows: {len(gold)}')
print(f'Classes: {gold["category_id_final"].nunique()}')

# Agreement between manual labels and rule-based labels
agree = (gold['category_id_rule'].fillna('CAT_UNKNOWN') == gold['category_id_final']).mean()
print(f'Agreement rate (manual vs rule-based): {agree:.1%}')

dist = gold['category_id_final'].value_counts().rename(index=CATEGORY_LABELS)
dist

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
dist.sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Annotated contracts')
ax.set_title('Gold label distribution (manual annotation)')
plt.tight_layout()
plt.savefig(FIGURES / 'annotation_distribution.png', dpi=150)
plt.show()

In [ ]:
# Rule-based vs. manual annotation disagreements
disagree = gold[gold['category_id_rule'].fillna('CAT_UNKNOWN') != gold['category_id_final']]
print(f'Disagreements: {len(disagree)} / {len(gold)} ({len(disagree)/len(gold):.1%})')
if len(disagree) > 0:
    print('\nTop disagreement pairs (rule_label → manual_label):')
    pairs = (
        disagree.groupby(['category_id_rule', 'category_id_final'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(10)
    )
    print(pairs.to_string(index=False))

## 2. Baseline Classifier (TF-IDF)

In [ ]:
baseline_results = pd.read_csv(TABLES / 'baseline_results.csv')
baseline_per_class = pd.read_csv(TABLES / 'baseline_per_class.csv')

print('=== Baseline CV results ===')
print(baseline_results.to_string(index=False))

In [ ]:
# Per-class F1
per_class = baseline_per_class[
    baseline_per_class['class'].isin(CATEGORY_LABELS.keys())
].copy()
per_class['label'] = per_class['class'].map(CATEGORY_LABELS)

fig, ax = plt.subplots(figsize=(9, 4))
per_class.sort_values('f1-score').plot.barh(
    x='label', y='f1-score', ax=ax, color='steelblue', legend=False
)
ax.axvline(0.70, color='red', linestyle='--', linewidth=1, label='F1=0.70 target')
ax.set_xlabel('F1-score')
ax.set_title('TF-IDF Baseline — per-class F1 (5-fold CV)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'baseline_per_class_f1.png', dpi=150)
plt.show()

In [ ]:
# Confusion matrix heatmap
cm_df = pd.read_csv(TABLES / 'baseline_confusion_matrix.csv', index_col=0)
cat_order = [c for c in CATEGORY_LABELS if c in cm_df.index]
cm_df = cm_df.reindex(index=cat_order, columns=cat_order)
labels = [CATEGORY_LABELS.get(c, c) for c in cat_order]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm_df, annot=True, fmt='d', cmap='Blues',
    xticklabels=labels, yticklabels=labels, ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('TF-IDF Baseline — confusion matrix (5-fold CV)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES / 'baseline_confusion_matrix.png', dpi=150)
plt.show()

## 3. Model Comparison (TF-IDF vs SBERT)

In [ ]:
comparison_path = TABLES / 'model_comparison.csv'
if comparison_path.exists():
    comp = pd.read_csv(comparison_path)
    print('=== Model Comparison ===')
    print(comp.to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(6, 3))
    colors = ['steelblue', 'coral']
    bars = ax.barh(comp['model'], comp['macro_f1'], color=colors)
    ax.axvline(0.70, color='red', linestyle='--', linewidth=1, label='F1=0.70 target')
    for bar, val in zip(bars, comp['macro_f1']):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10)
    ax.set_xlabel('Macro F1')
    ax.set_title('TF-IDF vs SBERT — macro F1 (5-fold CV)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES / 'model_comparison.png', dpi=150)
    plt.show()
else:
    print('model_comparison.csv not found — run nlp_advanced_classifier.py first')

## 4. Full Corpus Propagation Results

In [ ]:
nlp_df = pd.read_csv('../data/processed/boamp_full_clean_nlp.csv', low_memory=False)
print(f'Full corpus: {len(nlp_df)} rows')
print(f'NLP columns: {[c for c in nlp_df.columns if "nlp" in c]}')

low_conf = nlp_df['nlp_low_confidence'].sum()
print(f'Low-confidence (<0.70): {low_conf} ({100*low_conf/len(nlp_df):.1f}%)')
print()

# NLP vs rule-based label agreement on full corpus
agree_full = (nlp_df['category_id'] == nlp_df['category_id_nlp']).mean()
print(f'Agreement (NLP vs rule-based, full corpus): {agree_full:.1%}')

In [ ]:
# Category distribution: rule-based vs NLP
rule_dist = nlp_df['category_id'].value_counts().rename('rule_based')
nlp_dist = nlp_df['category_id_nlp'].value_counts().rename('nlp_model')
dist_compare = pd.concat([rule_dist, nlp_dist], axis=1).fillna(0).astype(int)
dist_compare.index = [CATEGORY_LABELS.get(i, i) for i in dist_compare.index]

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(dist_compare))
width = 0.4
ax.bar([i - width/2 for i in x], dist_compare['rule_based'], width, label='Rule-based', color='steelblue')
ax.bar([i + width/2 for i in x], dist_compare['nlp_model'], width, label='NLP model', color='coral')
ax.set_xticks(list(x))
ax.set_xticklabels(dist_compare.index, rotation=45, ha='right')
ax.set_ylabel('Number of contracts')
ax.set_title('Category distribution — rule-based vs NLP model (full corpus)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'category_distribution_comparison.png', dpi=150)
plt.show()

In [ ]:
# Confidence histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(nlp_df['nlp_confidence'], bins=40, color='steelblue', edgecolor='white')
ax.axvline(0.70, color='red', linestyle='--', linewidth=1.5, label='Confidence threshold (0.70)')
ax.set_xlabel('Max predicted probability')
ax.set_ylabel('Contracts')
ax.set_title('NLP model confidence distribution (full corpus)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'nlp_confidence_histogram.png', dpi=150)
plt.show()

In [ ]:
# Low-confidence by category
low_by_cat = (
    nlp_df.groupby('category_id_nlp')['nlp_low_confidence']
    .agg(['sum', 'count'])
    .assign(pct=lambda d: 100 * d['sum'] / d['count'])
    .rename(columns={'sum': 'low_conf_count', 'count': 'total'})
    .sort_values('pct', ascending=False)
)
low_by_cat.index = [CATEGORY_LABELS.get(i, i) for i in low_by_cat.index]
print('Low-confidence rate by category:')
print(low_by_cat[['total', 'low_conf_count', 'pct']].to_string())

## 5. Deliverable L2 Summary

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Reload baseline results
br = pd.read_csv(TABLES / 'baseline_results.csv')
macro_f1 = br['macro_f1_mean'].max()

# Kappa from results file
kappa = br['kappa_rule_vs_gold'].iloc[0] if 'kappa_rule_vs_gold' in br.columns else 'N/A'

# SBERT comparison
comp_path = TABLES / 'model_comparison.csv'
if comp_path.exists():
    comp = pd.read_csv(comp_path)
    sbert_f1 = comp.loc[comp['model'].str.contains('SBERT'), 'macro_f1'].values
    sbert_f1 = sbert_f1[0] if len(sbert_f1) else 'N/A'
else:
    sbert_f1 = 'not run'

print('=' * 50)
print('DELIVERABLE L2 — NLP Classification Summary')
print('=' * 50)
print(f'Annotated contracts:       {len(gold)}')
print(f'Manual vs rule-based agree: {agree:.1%}')
print(f"Cohen's kappa:              {kappa}")
print(f'TF-IDF macro F1 (5-fold):  {macro_f1:.4f}')
print(f'SBERT macro F1 (5-fold):   {sbert_f1}')
print(f'Full corpus propagated:    {len(nlp_df)} rows')
print(f'Low-confidence flagged:    {low_conf} ({100*low_conf/len(nlp_df):.1f}%)')
print(f'Output: boamp_full_clean_nlp.csv')
print(f'Output: boamp_phase2_survival_nlp.csv')
print('=' * 50)